<a href="https://colab.research.google.com/github/reinaldoyungh/Python/blob/main/IA_Avaliacao_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Carregar um arquivo .csv com feedback de clientes (reviews.csv)

2. Usar um LLM para classificar o sentimento de cada feedback (Positivo, Negativo, Neutro)

3. Adicionar uma nova coluna com essa classificação ao DataFrame.

In [6]:
!pip install -q groq
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
from groq import Groq
client = Groq()

import pandas as pd

In [7]:

def chama_IA(perguntas):
    all_responses = []
    for pergunta_individual in perguntas:
        response_stream = client.chat.completions.create(
          model="openai/gpt-oss-20b",
          messages=[
            {
              "role": "user",
              "content": pergunta_individual
            }
          ],
          temperature=1,
          max_completion_tokens=1024,
          top_p=1,
          reasoning_effort="medium",
          stream=True,
          stop=None
        )

        collected_text = ""
        for chunk in response_stream:
            if chunk.choices and chunk.choices[0].delta.content:
                collected_text += chunk.choices[0].delta.content
        all_responses.append(collected_text)
    return all_responses

In [ ]:
lista_reviews = []
with open('reviews.csv', 'r') as arquivo:
  for linha in arquivo:
    lista_reviews.append(linha.strip())

In [9]:
import io

# Convert the list of CSV strings into a pandas DataFrame
df_reviews = pd.read_csv(io.StringIO('\n'.join(lista_reviews)))

# Display the 'reviewerName' and 'reviewText' columns
display(df_reviews[['reviewerName', 'reviewText']])




,reviewerName,reviewText
0,1K3,it works as expected. I should have sprung for...
1,1m2,This think has worked out great.Had a diff. br...
2,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit..."
3,2Cents!,It's mini storage. It doesn't do anything els...
4,2K1Toaster,I have it in my phone and it never skips a bea...
5,98020,"It works, but file writes are a bit slower tha..."
6,"Abdulrahman J. Alrashed ""dr34m3r""","I bought 2 of those SanDisk 32 GB microSD , us..."
7,"Abel Feliciano ""Ace Master""",The memory card is an excellent condition and ...
8,Abraham Arturo Meza Marin,I bougth this micro SD card after some trubles...
9,"Abused Commuter ""abused_commuter""",Ordered this for a Galaxy S3. Lasted a few mo...


In [12]:
reviews_to_classify = df_reviews['reviewText'].tolist()

# Prepare questions for the LLM to classify sentiment
perguntas_sentimento = [
    f"Classifique o sentimento da seguinte avaliação do cliente como Positivo, Negativo ou Neutro: {review}"
    for review in reviews_to_classify
]

# Call the LLM function to get sentiments
sentimentos = chama_IA(perguntas_sentimento)

# Add the sentiments as a new column to the DataFrame
df_reviews['sentimento'] = sentimentos

# Display the DataFrame with the new sentiment column
display(df_reviews[['reviewerName', 'reviewText', 'sentimento']].head())

,reviewerName,reviewText,sentimento
0,1K3,it works as expected. I should have sprung for...,Negativo
1,1m2,This think has worked out great.Had a diff. br...,Positivo
2,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",Positivo
3,2Cents!,It's mini storage. It doesn't do anything els...,Positivo
4,2K1Toaster,I have it in my phone and it never skips a bea...,**Positivo**


In [13]:
df_reviews.to_csv('reviews_avaliados.csv', index=False)
print('DataFrame salvo como reviews_avaliados.csv')

DataFrame salvo como reviews_avaliados.csv
